In [1]:
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.4 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login

hf_token = "YOUR_TOKEN_HERE"

login(token=hf_token)
print("Successfully logged into Hugging Face Hub!")

Successfully logged into Hugging Face Hub!


In [3]:

from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="ai4bharat/indicwav2vec_v1_bengali")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/940 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

In [4]:
!pip install -q datasets huggingface_hub

from datasets import load_dataset

bengali_valid = load_dataset(
    "ai4bharat/IndicVoices",
    "bengali",
    split="valid",        # the split name inside the config; usually “train” for the data split
    streaming=True       # streams rows on‑demand
)


README.md:   0%|          | 0.00/38.8k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

In [5]:
next(iter(bengali_valid))

{'audio_filepath': <datasets.features._torchcodec.AudioDecoder at 0x7e76dc6ed550>,
 'text': 'স্কুলের পর টিউশন প্রাইভেট না পড়ে সন্ধ্যার দিকে স্কুল টিউশন প্রাইভেট পড়া খুব',
 'duration': 6.026,
 'lang': 'bn',
 'samples': 96416,
 'verbatim': 'ইস্কুলের পর টিউশন পাইভেট না পড়ে সন্ধ্যার দিকে ইস্কুল টিউশন পাইভেট পড়া খুব',
 'normalized': 'স্কুলের পর টিউশন প্রাইভেট না পড়ে সন্ধ্যার দিকে স্কুল টিউশন প্রাইভেট পড়া খুব',
 'speaker_id': 'S4257107800381118',
 'scenario': 'Extempore',
 'task_name': 'DOI - Education',
 'gender': 'Male',
 'age_group': '18-30',
 'job_type': 'Student',
 'qualification': 'Upto 12th',
 'area': 'Rural',
 'district': 'Jhargram',
 'state': 'West Bengal',
 'occupation': 'Student',
 'verification_report': "{'decision': 'excellent', 'low_volume': False, 'noise_intermittent': False, 'chatter_intermittent': False, 'noise_persistent': False, 'chatter_persistent': False, 'unclear_audio': False, 'off_topic': False, 'repeating_content': False, 'long_pauses': False, 'mispronunciation

In [5]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 25.6 MB/s eta 0:00:00


In [ ]:
if hasattr(bengali_valid, 'info') and 'valid' in bengali_valid.info.splits:
    total_samples_in_dataset = bengali_valid.info.splits['valid'].num_examples
    print(f"The total number of samples in the 'bengali_valid' dataset (according to dataset metadata) is: {total_samples_in_dataset}")
else:
    print("Could not retrieve the total number of samples from dataset info. This might be a truly streaming dataset without a known total count.")


The total number of samples in the 'bengali_valid' dataset (according to dataset metadata) is: 3906


In [ ]:
import time
from tqdm.auto import tqdm


predictions_first_half = []
references_first_half = []

print(f"Starting ASR inference on {total_samples_in_dataset} samples)...")


for i, sample in tqdm(enumerate(bengali_valid), total=total_samples_in_dataset, desc="Processing Bengali_valid"):
    start_time = time.time() # Start timer for the current sample

    audio_data = sample["audio_filepath"]
    audio_array = audio_data["array"]
    sampling_rate = audio_data["sampling_rate"]
    reference_text = sample["text"]


    transcription = pipe({"sampling_rate": sampling_rate, "raw": audio_array})['text']


    predictions_first_half.append(transcription)
    references_first_half.append(reference_text)

    end_time = time.time() # End timer for the current sample
    iteration_duration = end_time - start_time


    tqdm.write(f"  Sample {i + 1}/{total_samples_in_dataset} processed in {iteration_duration:.4f} seconds.")

print("Inference for Bengali_valid complete.")

Starting ASR inference on 3906 samples)...


Processing Bengali_valid:   0%|          | 0/3906 [00:00<?, ?it/s]

  Sample 1/3906 processed in 18.1698 seconds.
  Sample 2/3906 processed in 6.1493 seconds.
  Sample 3/3906 processed in 6.4120 seconds.
  Sample 4/3906 processed in 8.9238 seconds.
  Sample 5/3906 processed in 12.9083 seconds.
  Sample 6/3906 processed in 6.6322 seconds.
  Sample 7/3906 processed in 11.7022 seconds.
  Sample 8/3906 processed in 4.9874 seconds.
  Sample 9/3906 processed in 6.8025 seconds.
  Sample 10/3906 processed in 11.7683 seconds.
  Sample 11/3906 processed in 9.4209 seconds.
  Sample 12/3906 processed in 0.4675 seconds.
  Sample 13/3906 processed in 2.5113 seconds.
  Sample 14/3906 processed in 6.2440 seconds.
  Sample 15/3906 processed in 3.0621 seconds.
  Sample 16/3906 processed in 3.4810 seconds.
  Sample 17/3906 processed in 0.5440 seconds.
  Sample 18/3906 processed in 3.5610 seconds.
  Sample 19/3906 processed in 1.6469 seconds.
  Sample 20/3906 processed in 2.7439 seconds.
  Sample 21/3906 processed in 1.3945 seconds.
  Sample 22/3906 processed in 1.5343 se

KeyboardInterrupt: 

In [ ]:
from jiwer import process_words,process_characters

#Word level
total_substitutions_words = 0
total_deletions_words = 0
total_insertions_words = 0
total_reference_words = 0

#Character level
total_substitutions_characters = 0
total_deletions_characters = 0
total_insertions_characters = 0
total_reference_characters = 0

for ref, pred in zip(references_first_half, predictions_first_half):
    metrics = process_words([ref], [pred])
    total_substitutions_words += metrics.substitutions
    total_deletions_words += metrics.deletions
    total_insertions_words+= metrics.insertions
    total_reference_words+= len(ref.split())

for ref, pred in zip(references_first_half, predictions_first_half):
    metrics = process_characters([ref], [pred])
    total_substitutions_characters += metrics.substitutions
    total_deletions_characters += metrics.deletions
    total_insertions_characters+= metrics.insertions
    total_reference_characters+= len(ref) # Corrected to count characters



print("\n--- Model Results(Word-level) for Set 1---")
print(f"Total reference words: {total_reference_words}")
print(f"Total substitutions: {total_substitutions_words}")
print(f"Total deletions: {total_deletions_words}")
print(f"Total insertions: {total_insertions_words}")

print("\n--- Model Results(Char-level) for Set 1---")
print(f"Total reference chars: {total_reference_characters}")
print(f"Total substitutions: {total_substitutions_characters}")
print(f"Total deletions: {total_deletions_characters}")
print(f"Total insertions: {total_insertions_characters}")


--- Model Results(Word-level) for Set 1---
Total reference words: 17567
Total substitutions: 5077
Total deletions: 2769
Total insertions: 133

--- Model Results(Char-level) for Set 1---
Total reference chars: 98076
Total substitutions: 5210
Total deletions: 12906
Total insertions: 1506


In [8]:
import time
from tqdm.auto import tqdm

subset2_bengali_valid=bengali_valid.skip(1165).take(2741)

predictions_second_half = []
references_second_half = []

print(f"Starting ASR inference on {2741} samples)...")


for i, sample in tqdm(enumerate(subset2_bengali_valid), total=2741, desc="Processing subset2_bengali_valid"):
    start_time = time.time() # Start timer for the current sample

    audio_data = sample["audio_filepath"]
    audio_array = audio_data["array"]
    sampling_rate = audio_data["sampling_rate"]
    reference_text = sample["text"]


    transcription = pipe({"sampling_rate": sampling_rate, "raw": audio_array})['text']


    predictions_second_half.append(transcription)
    references_second_half.append(reference_text)

    end_time = time.time() # End timer for the current sample
    iteration_duration = end_time - start_time


    tqdm.write(f"  Sample {i + 1}/{2741} processed in {iteration_duration:.4f} seconds.")

print("Inference for subset2_bengali_valid complete.")

Starting ASR inference on 2741 samples)...


Processing subset2_bengali_valid:   0%|          | 0/2741 [00:00<?, ?it/s]

  Sample 1/2741 processed in 26.6078 seconds.
  Sample 2/2741 processed in 5.8340 seconds.
  Sample 3/2741 processed in 15.0163 seconds.
  Sample 4/2741 processed in 10.4829 seconds.
  Sample 5/2741 processed in 7.2298 seconds.
  Sample 6/2741 processed in 13.2309 seconds.
  Sample 7/2741 processed in 0.5616 seconds.
  Sample 8/2741 processed in 0.6695 seconds.
  Sample 9/2741 processed in 0.7250 seconds.
  Sample 10/2741 processed in 0.6233 seconds.
  Sample 11/2741 processed in 0.6487 seconds.
  Sample 12/2741 processed in 0.5859 seconds.
  Sample 13/2741 processed in 0.4700 seconds.
  Sample 14/2741 processed in 0.4739 seconds.
  Sample 15/2741 processed in 0.5025 seconds.
  Sample 16/2741 processed in 0.4188 seconds.
  Sample 17/2741 processed in 0.4457 seconds.
  Sample 18/2741 processed in 10.4101 seconds.
  Sample 19/2741 processed in 2.2246 seconds.
  Sample 20/2741 processed in 3.2543 seconds.
  Sample 21/2741 processed in 3.2688 seconds.
  Sample 22/2741 processed in 0.5352 s

KeyboardInterrupt: 

In [9]:
from jiwer import process_words,process_characters

#Word level
total_substitutions_words = 0
total_deletions_words = 0
total_insertions_words = 0
total_reference_words = 0

#Character level
total_substitutions_characters = 0
total_deletions_characters = 0
total_insertions_characters = 0
total_reference_characters = 0

for ref, pred in zip(references_second_half, predictions_second_half):
    metrics = process_words([ref], [pred])
    total_substitutions_words += metrics.substitutions
    total_deletions_words += metrics.deletions
    total_insertions_words+= metrics.insertions
    total_reference_words+= len(ref.split())

for ref, pred in zip(references_second_half, predictions_second_half):
    metrics = process_characters([ref], [pred])
    total_substitutions_characters += metrics.substitutions
    total_deletions_characters += metrics.deletions
    total_insertions_characters+= metrics.insertions
    total_reference_characters+= len(ref) # Corrected to count characters



print("\n--- Model Results(Word-level) for Set 2---")
print(f"Total reference words: {total_reference_words}")
print(f"Total substitutions: {total_substitutions_words}")
print(f"Total deletions: {total_deletions_words}")
print(f"Total insertions: {total_insertions_words}")

print("\n--- Model Results(Char-level) for Set 2---")
print(f"Total reference chars: {total_reference_characters}")
print(f"Total substitutions: {total_substitutions_characters}")
print(f"Total deletions: {total_deletions_characters}")
print(f"Total insertions: {total_insertions_characters}")


--- Model Results(Word-level) for Set 2---
Total reference words: 20805
Total substitutions: 6138
Total deletions: 3930
Total insertions: 155

--- Model Results(Char-level) for Set 2---
Total reference chars: 113817
Total substitutions: 6445
Total deletions: 17899
Total insertions: 1594


In [6]:
import time
from tqdm.auto import tqdm

subset3_bengali_valid=bengali_valid.skip(2891).take(1015)

predictions_third_half = []
references_third_half = []

print(f"Starting ASR inference on {1015} samples)...")


for i, sample in tqdm(enumerate(subset3_bengali_valid), total=1015, desc="Processing subset3_bengali_valid"):
    start_time = time.time() # Start timer for the current sample

    audio_data = sample["audio_filepath"]
    audio_array = audio_data["array"]
    sampling_rate = audio_data["sampling_rate"]
    reference_text = sample["text"]


    transcription = pipe({"sampling_rate": sampling_rate, "raw": audio_array})['text']


    predictions_third_half.append(transcription)
    references_third_half.append(reference_text)

    end_time = time.time() # End timer for the current sample
    iteration_duration = end_time - start_time


    tqdm.write(f"  Sample {i + 1}/{1015} processed in {iteration_duration:.4f} seconds.")

print("Inference for subset3_bengali_valid complete.")

Starting ASR inference on 1015 samples)...


Processing subset3_bengali_valid:   0%|          | 0/1015 [00:00<?, ?it/s]

  Sample 1/1015 processed in 2.5353 seconds.
  Sample 2/1015 processed in 0.2935 seconds.
  Sample 3/1015 processed in 0.1654 seconds.
  Sample 4/1015 processed in 0.0717 seconds.
  Sample 5/1015 processed in 0.1209 seconds.
  Sample 6/1015 processed in 0.1004 seconds.
  Sample 7/1015 processed in 0.0770 seconds.
  Sample 8/1015 processed in 0.0812 seconds.


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Sample 9/1015 processed in 0.1298 seconds.
  Sample 10/1015 processed in 0.0633 seconds.
  Sample 11/1015 processed in 0.0710 seconds.
  Sample 12/1015 processed in 0.1005 seconds.
  Sample 13/1015 processed in 0.0499 seconds.
  Sample 14/1015 processed in 0.0746 seconds.
  Sample 15/1015 processed in 0.0629 seconds.
  Sample 16/1015 processed in 0.1524 seconds.
  Sample 17/1015 processed in 0.1830 seconds.
  Sample 18/1015 processed in 0.1779 seconds.
  Sample 19/1015 processed in 0.0547 seconds.
  Sample 20/1015 processed in 0.2047 seconds.
  Sample 21/1015 processed in 0.0538 seconds.
  Sample 22/1015 processed in 0.1116 seconds.
  Sample 23/1015 processed in 0.1850 seconds.
  Sample 24/1015 processed in 0.0625 seconds.
  Sample 25/1015 processed in 0.0727 seconds.
  Sample 26/1015 processed in 0.1156 seconds.
  Sample 27/1015 processed in 0.1121 seconds.
  Sample 28/1015 processed in 0.0781 seconds.
  Sample 29/1015 processed in 0.1909 seconds.
  Sample 30/1015 processed in 0.060

In [7]:
from jiwer import process_words,process_characters

#Word level
total_substitutions_words = 0
total_deletions_words = 0
total_insertions_words = 0
total_reference_words = 0

#Character level
total_substitutions_characters = 0
total_deletions_characters = 0
total_insertions_characters = 0
total_reference_characters = 0

for ref, pred in zip(references_third_half, predictions_third_half):
    metrics = process_words([ref], [pred])
    total_substitutions_words += metrics.substitutions
    total_deletions_words += metrics.deletions
    total_insertions_words+= metrics.insertions
    total_reference_words+= len(ref.split())

for ref, pred in zip(references_third_half, predictions_third_half):
    metrics = process_characters([ref], [pred])
    total_substitutions_characters += metrics.substitutions
    total_deletions_characters += metrics.deletions
    total_insertions_characters+= metrics.insertions
    total_reference_characters+= len(ref) # Corrected to count characters



print("\n--- Model Results(Word-level) for Set 3---")
print(f"Total reference words: {total_reference_words}")
print(f"Total substitutions: {total_substitutions_words}")
print(f"Total deletions: {total_deletions_words}")
print(f"Total insertions: {total_insertions_words}")

print("\n--- Model Results(Char-level) for Set 3---")
print(f"Total reference chars: {total_reference_characters}")
print(f"Total substitutions: {total_substitutions_characters}")
print(f"Total deletions: {total_deletions_characters}")
print(f"Total insertions: {total_insertions_characters}")


--- Model Results(Word-level) for Set 3---
Total reference words: 14776
Total substitutions: 4091
Total deletions: 2398
Total insertions: 139

--- Model Results(Char-level) for Set 3---
Total reference chars: 82277
Total substitutions: 4212
Total deletions: 11626
Total insertions: 1199
